# 5.3 poskyris. Rezultatai su VoiceBank + DEMAND duomenų rinkiniu

## 1. Importai ir nustatymai

In [ ]:
# 1. Importai ir nustatymai
import pandas as pd
import numpy as np
import json
import os
import csv
import warnings
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        9,
    'axes.titlesize':   11,
    'axes.labelsize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'figure.dpi':       150,
})

CSV_PATH     = 'results/all_runs.csv'
SUMMARY_DIR  = 'results'
FIGURES_DIR  = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

EXP_TARGET = 'full_voicebank_unetdilatedmaskcnn_v2'

def lt_num(v, dec=3):
    return f'{v:.{dec}f}'.replace('.', ',')

print("Nustatymai įkelti. Tikslinis eksperimentas:", EXP_TARGET)


## 2. Duomenu nuskaitymas ir diagnostika

In [ ]:
# 2. Duomenų nuskaitymas ir diagnostika
df = pd.read_csv(CSV_PATH)

print("=== Stulpeliai ===")
print(list(df.columns))
print()
print(f"Iš viso eilučių: {len(df)}")
print()
print("=== Pirmosios 3 eilutės ===")
display(df.head(3))
print()
print("=== Visi eksperimentų pavadinimai ===")
for n in sorted(df['experiment_id'].unique()):
    print(" ", n)
print()
vb_mask = df['experiment_id'].str.lower().str.contains('voicebank')
print("=== VoiceBank eksperimentai ===")
display(df[vb_mask][['experiment_id', 'dataset_name', 'model_name',
                       'loss_name', 'learning_rate', 'n_fft', 'segment_seconds']])


## 3. VoiceBank eksperimento radimas

In [ ]:
# 3. VoiceBank eksperimento radimas
rows = df[df['experiment_id'] == EXP_TARGET]

if len(rows) == 0:
    raise RuntimeError(f"Eksperimentas '{EXP_TARGET}' nerastas all_runs.csv!")
if len(rows) > 1:
    print(f"Rasta {len(rows)} eilutes. Naudojama paskutine.")
    row = rows.iloc[-1]
else:
    row = rows.iloc[0]

print(f"Rastas eksperimentas: {row['experiment_id']}")
print(f"  Modelis:          {row['model_name']}")
print(f"  Nuostoliu f-ja:   {row['loss_name']}")
print(f"  Mokymosi greitis: {row['learning_rate']}")
print(f"  STFT langas:      {row['n_fft']}")
print(f"  Segmentas, s:     {row['segment_seconds']}")


## 4. Reikšmių ištraukimas ir patikra

In [ ]:
# 4. Reikšmių ištraukimas ir patikra
pesq_before  = row['mean_pesq_noisy']
pesq_after   = row['mean_pesq_enhanced']
delta_pesq   = row['mean_delta_pesq']
stoi_before  = row['mean_stoi_noisy']
stoi_after   = row['mean_stoi_enhanced']
delta_stoi   = row['mean_delta_stoi']
delta_snr    = row['mean_delta_snr_est']
snr_before   = row['mean_snr_noisy_est']
snr_after    = row['mean_snr_enhanced_est']
n_files      = int(row['num_eval_files'])
best_val     = row['best_val_loss']
best_epoch   = int(row['best_epoch'])
epochs_done  = int(row['epochs_completed'])
total_epochs = int(row['epochs'])

if 'num_skipped_files' in row and not pd.isna(row['num_skipped_files']):
    n_skipped = int(row['num_skipped_files'])
else:
    n_skipped = None
    print("'num_skipped_files' nerastas arba NaN.")

json_path = os.path.join(SUMMARY_DIR, EXP_TARGET, 'run_summary.json')
if os.path.exists(json_path):
    with open(json_path, encoding='utf-8') as f:
        summary = json.load(f)
    print(f"run_summary.json rastas: {json_path}")
    if n_skipped is None:
        n_skipped = summary.get('num_skipped_files', 0)
else:
    print(f"Ispejimas: {json_path} nerastas.")
    summary = {}

print()
print("=== IsTrauktos reikšmes ===")
print(f"  PESQ pries apdorojima:  {lt_num(pesq_before)}")
print(f"  PESQ po apdorojimo:     {lt_num(pesq_after)}")
print(f"  DPESQ:                  {lt_num(delta_pesq)}")
print(f"  STOI pries apdorojima:  {lt_num(stoi_before)}")
print(f"  STOI po apdorojimo:     {lt_num(stoi_after)}")
print(f"  DSTOI:                  {lt_num(delta_stoi)}")
print(f"  DSNR, dB:               {lt_num(delta_snr, 2)}")
print(f"  Testavimo failai:       {n_files}")
print(f"  Praleisti failai:       {n_skipped if n_skipped is not None else 'N/A'}")
print(f"  Geriausias val. nuost.: {lt_num(best_val, 6)}")
print(f"  Geriausia epocha:       {best_epoch}")
print(f"  Epochu atlikta:         {epochs_done} / {total_epochs}")


## 5. Nuostolių kreivė


In [ ]:
# 5a. history.csv nuskaitymas ir diagnostika
HIST_PATH = os.path.join(SUMMARY_DIR, EXP_TARGET, 'history.csv')
JSON_PATH = os.path.join(SUMMARY_DIR, EXP_TARGET, 'run_summary.json')

if not os.path.exists(HIST_PATH):
    raise FileNotFoundError(f"Nerastas: {HIST_PATH}")

hist = pd.read_csv(HIST_PATH)
print("history.csv stulpeliai:", list(hist.columns))
print(f"Eiluciu (epochu): {len(hist)}")
display(hist.head(5))

# Geriausios epochos nustatymas is run_summary.json
if os.path.exists(JSON_PATH):
    with open(JSON_PATH, encoding='utf-8') as f:
        _sum = json.load(f)
    _best_ep    = int(_sum['best_epoch'])
    _max_epochs = int(_sum['epochs'])
    _ep_done    = int(_sum['epochs_completed'])
    print(f"run_summary.json: best_epoch={_best_ep}, "
          f"epochs={_max_epochs}, epochs_completed={_ep_done}")
else:
    print(f"Ispejimas: {JSON_PATH} nerastas.")
    if 'improved' in hist.columns:
        _best_ep = int(hist[hist['improved'] == True]['epoch'].max())
    else:
        _best_ep = int(hist.loc[hist['val_loss'].idxmin(), 'epoch'])
        print("'improved' stulpelio nera – naudojamas min(val_loss) epocha.")
    _max_epochs = None
    _ep_done    = int(hist['epoch'].max())
print(f"Geriausia epocha: {_best_ep}")


In [ ]:
_col_ep  = 'epoch'
_col_tr  = 'train_loss'
_col_val = 'val_loss'
for _c in [_col_ep, _col_tr, _col_val]:
    if _c not in hist.columns:
        raise KeyError(f"history.csv neturi stulpelio '{_c}'!")

_ep_vals  = hist[_col_ep].values
_tr_vals  = hist[_col_tr].values
_vl_vals  = hist[_col_val].values
_best_vl  = float(hist[hist[_col_ep] == _best_ep][_col_val].iloc[0])

fig, ax = plt.subplots(figsize=(7.5, 4.0))

ax.plot(_ep_vals, _tr_vals, color='#2171B5', linewidth=1.4,
        label='Mokymo nuostolis')
ax.plot(_ep_vals, _vl_vals, color='#EF6C00', linewidth=1.4,
        linestyle='--', label='Validavimo nuostolis')
ax.axvline(x=_best_ep, color='#388E3C', linewidth=1.2, linestyle=':',
           label=f'Geriausia epocha ({_best_ep})')
ax.scatter([_best_ep], [_best_vl], color='#388E3C', zorder=5, s=42)

ax.set_xlabel('Epocha', fontsize=10)
ax.set_ylabel('Nuostolis', fontsize=10)
ax.set_title('VoiceBank + DEMAND mokymo ir validavimo nuostoliu kaita',
             fontsize=10, fontweight='bold', pad=8)

ax.set_xlim(0, _ep_done + 1)
_ymax = max(_tr_vals.max(), _vl_vals.max()) * 1.08
ax.set_ylim(0, _ymax)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda v, _: f'{v:.4f}'.replace('.', ','))
)
ax.grid(linestyle='--', linewidth=0.5, alpha=0.55)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', fontsize=8.5, framealpha=0.85)

_note_max = str(_max_epochs) if _max_epochs else '?'
ax.annotate(
    f'Ankstyvasis stabdymas: {_ep_done} / {_note_max} epochu. '
    f'Geriausia val. nuost.: {_best_vl:.6f}'.replace('.', ',') + f' ({_best_ep} ep.)',
    xy=(0.01, 0.02), xycoords='axes fraction',
    fontsize=7.5, color='#555555', va='bottom', ha='left'
)

plt.tight_layout()

_png24 = os.path.join(FIGURES_DIR, '24_voicebank_loss_curve.png')
_svg24 = os.path.join(FIGURES_DIR, '24_voicebank_loss_curve.svg')
fig.savefig(_png24, dpi=300, bbox_inches='tight')
fig.savefig(_svg24, bbox_inches='tight')
plt.show()
print(f"Issaugota: {_png24}")
print(f"Issaugota: {_svg24}")


## 6. PESQ ir STOI rezultatai

In [ ]:
LABELS = ['Pries\napdorojima', 'Po\napdorojimo']
BAR_W  = 0.45
COLORS = ['#5B9BD5', '#70AD47']
EDGE   = '#333333'

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.8))

for ax, subtitle, vals, ylim_lo, ylim_hi in [
    (axes[0], 'PESQ', [pesq_before, pesq_after], 1.4,  2.8),
    (axes[1], 'STOI', [stoi_before, stoi_after], 0.88, 0.96),
]:
    x = np.arange(len(vals))
    bars = ax.bar(x, vals, width=BAR_W, color=COLORS,
                  edgecolor=EDGE, linewidth=0.8)
    pad = (ylim_hi - ylim_lo) * 0.015
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + pad,
                lt_num(v, 3),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(LABELS, fontsize=9)
    ax.set_ylabel(subtitle, fontsize=10)
    ax.set_ylim(ylim_lo, ylim_hi)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: lt_num(v, 2))
    )
    ax.set_title(subtitle, fontsize=11, fontweight='bold', pad=6)
    ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    'VoiceBank + DEMAND PESQ ir STOI rezultatai pries ir po apdorojimo',
    fontsize=10, y=1.01
)
plt.tight_layout()

_png25 = os.path.join(FIGURES_DIR, '25_voicebank_pesq_stoi_pries_po.png')
_svg25 = os.path.join(FIGURES_DIR, '25_voicebank_pesq_stoi_pries_po.svg')
fig.savefig(_png25, dpi=300, bbox_inches='tight')
fig.savefig(_svg25, bbox_inches='tight')
plt.show()
print(f"Issaugota: {_png25}")
print(f"Issaugota: {_svg25}")


## 7. CSV lentelių išsaugojimas

In [ ]:
# 7. CSV lentelių išsaugojimas

# --- 7a. Rezultatai ---
csv1_path = os.path.join(FIGURES_DIR, 'voicebank_rezultatai.csv')
rows1 = [
    ['Rodiklis', 'Pries apdorojima', 'Po apdorojimo', 'Pokytis'],
    ['PESQ',  lt_num(pesq_before, 3), lt_num(pesq_after, 3),  lt_num(delta_pesq, 3)],
    ['STOI',  lt_num(stoi_before, 3), lt_num(stoi_after, 3),  lt_num(delta_stoi, 3)],
    ['DSNR, dB', 'Netaikoma', 'Netaikoma', lt_num(delta_snr, 2)],
]
with open(csv1_path, 'w', newline='', encoding='utf-8-sig') as f:
    csv.writer(f).writerows(rows1)
print(f"Issaugota: {csv1_path}")

# --- 7b. Eksperimento santrauka ---
csv2_path = os.path.join(FIGURES_DIR, 'voicebank_eksperimento_santrauka.csv')
rows2 = [
    ['Eksperimentas', 'Modelis', 'Nuostoliu funkcija', 'Mokymosi greitis',
     'STFT lango dydis', 'Segmento trukme, s',
     'Geriausias validavimo nuostolis', 'Geriausia epocha',
     'Is viso epochu', 'Testavimo failu skaicius', 'Pralestu failu skaicius'],
    [EXP_TARGET, row['model_name'], row['loss_name'],
     str(row['learning_rate']), str(int(row['n_fft'])), str(int(row['segment_seconds'])),
     lt_num(best_val, 6), str(best_epoch), str(epochs_done),
     str(n_files), str(n_skipped if n_skipped is not None else 0)],
]
with open(csv2_path, 'w', newline='', encoding='utf-8-sig') as f:
    csv.writer(f).writerows(rows2)
print(f"Issaugota: {csv2_path}")

# --- 7c. Mokymo santrauka ---
_has_lr   = 'learning_rate' in hist.columns
_final_lr = float(hist.iloc[-1]['learning_rate']) if _has_lr else None
_last_tr  = float(hist.iloc[-1]['train_loss'])
_last_vl  = float(hist.iloc[-1]['val_loss'])

csv3_path = os.path.join(FIGURES_DIR, 'voicebank_mokymo_santrauka.csv')
rows3 = [
    ['Eksperimentas', 'Maksimalus epochu skaicius', 'Faktiskai mokyta epochu',
     'Geriausia epocha', 'Geriausias validavimo nuostolis',
     'Paskutines epochos mokymo nuostolis',
     'Paskutines epochos validavimo nuostolis',
     'Galutinis mokymosi greitis'],
    [EXP_TARGET,
     str(_max_epochs) if _max_epochs else 'Nerasta',
     str(_ep_done),
     str(_best_ep),
     lt_num(_best_vl, 6),
     lt_num(_last_tr, 6),
     lt_num(_last_vl, 6),
     lt_num(_final_lr, 6) if _final_lr is not None else 'Nerasta'],
]
with open(csv3_path, 'w', newline='', encoding='utf-8-sig') as f:
    csv.writer(f).writerows(rows3)
print(f"Issaugota: {csv3_path}")

print()
display(pd.read_csv(csv1_path,  encoding='utf-8-sig'))
display(pd.read_csv(csv3_path,  encoding='utf-8-sig').T.rename(columns={0: 'Reiksme'}))
